# Xiexie wake-word — openWakeWord training notebook

> **Last verified working on Colab H100 / T4, Python 3.12.13 — May 5 2026.**

Trains a custom **"Xiexie"** wake-word detector and exports `xiexie.onnx` to
your Google Drive. Each cell is annotated with its purpose and an estimated
wall-clock time on a Colab **H100** (Pro/Pro+) — T4 timings are roughly
3–5× longer.

## What this notebook does

1. Installs `piper-sample-generator>=3.2.0` (PyPI), which bundles the new
   `piper-tts==1.4.x` with **espeak-ng embedded** — sidesteps the dead
   `piper-phonemize` wheel situation on Python 3.12.
2. Clones `openWakeWord` HEAD and applies a one-line `setup.py` patch to
   drop `speexdsp-ns` (no py3.12 wheel; only used at runtime, not training).
3. Installs the training extras **without** TensorFlow / `onnx_tf` —
   we never call `--convert_to_tflite`; the runtime in
   `backend/xiexie/voice/wake.py` consumes ONNX directly.
4. Downloads MIT room-impulse-responses + an AudioSet shard + 1 h of FMA
   music for noise / reverb augmentation.
5. Downloads ~2 000 h of pre-computed openWakeWord features (ACAV100M)
   plus an ~11 h validation set.
6. Generates ~N synthetic positives ("Xiexie", multi-speaker LibriTTS) +
   adversarial negatives via a **shim file** that injects the LibriTTS
   model path into the modern `piper_sample_generator.generate_samples`.
7. Mixes in the user's local recordings from `positives.zip` (oversampled).
8. Augments → extracts features → trains the small DNN classifier on top
   of the frozen Google speech-embedding backbone.
9. Evaluates (recall on held-out positives, false-trigger rate on noise).
10. Copies `xiexie.onnx` to `MyDrive/xiexie/xiexie.onnx`.

## Approximate H100 wall-clock

| Stage | H100 |
|---|---|
| Install + clone + patch | ~6 min |
| Background-audio downloads | ~6 min |
| Pre-computed-features download (3 GB) | ~2 min |
| Synthetic-clip generation | ~5 min |
| Augment + feature extraction | ~15 min |
| Train (early-stops at target) | ~30–60 min |
| Eval + export | ~1 min |
| **Total** | **~60–90 min** |

## Required Colab runtime

**Runtime → Change runtime type → H100 GPU, High-RAM ON, Python 3 (3.12 default).**
T4 also works (just slower). CPU also works (~6 h, painful — avoid).


## 0 — Configuration

Tweak everything from this single cell. Defaults target a ~75-min H100 run.


In [ ]:
CONFIG = {
    # --- core ---
    "wake_word": "Xiexie",
    "model_name": "xiexie",

    # --- dataset sizes ---
    # synthetic positives generated by Piper. The libritts multi-speaker
    # generator gives accent/prosody diversity for free; bumping these up
    # mostly helps if the trained model under-fires on the demo mic.
    "n_samples_train": 2000,
    "n_samples_val": 500,
    # how many copies of each user recording we duplicate into positive_train
    # (real recordings are scarce + high-signal so we oversample them).
    "user_recording_duplication": 8,

    # --- training ---
    "steps": 15000,
    "layer_size": 32,
    "target_accuracy": 0.7,
    "target_recall": 0.5,
    "target_false_positives_per_hour": 0.2,
    "max_negative_weight": 1500,
    # Piper TTS batch — H100 80 GB easily fits 100; T4 16 GB use 32.
    "tts_batch_size": 64,

    # --- background data (more = more robust, but more disk + time) ---
    # AudioSet shard `bal_train09.tar` is ~1.5 GB; keep it for default runs.
    "audioset_shard": "bal_train09.tar",
    "fma_hours": 1,

    # --- where to put the final model on Drive ---
    "drive_output_dir": "/content/drive/MyDrive/xiexie",
    # path inside Drive where you'll upload the recorded positives zip.
    # if missing the run still works (synthetic-only positives).
    "drive_positives_zip": "/content/drive/MyDrive/xiexie/positives.zip",
}
CONFIG


## 1a — System packages (~10 s on H100)

`espeak-ng` ships with the modern `piper-tts` wheel as a baked-in libary,
so installing the apt package is **belt-and-braces** in case the
runtime's libc fights the bundled binary. No-op if already present.


In [ ]:
!apt-get -qq update && apt-get -qq install -y espeak-ng > /dev/null
!espeak-ng --version | head -1


## 1b — Python deps: piper-sample-generator + training extras (~3 min on H100)

We **do not** install `tensorflow-cpu==2.8.1`, `tensorflow_probability` or
`onnx_tf` — those were only ever needed for the opt-in
`--convert_to_tflite` flag, which we never pass. The runtime
(`backend/xiexie/voice/wake.py`) loads the model via `onnxruntime`.

`piper-sample-generator>=3.2,<4` (PyPI, 2026-03-12) replaces the old
`pip install piper-phonemize` — the new generator pulls in
`piper-tts==1.4.x` with **espeak-ng embedded in the wheel**, so the
dead `piper-phonemize` cp310-only wheels no longer matter.


In [ ]:
!pip install -q -U pip

!pip install -q "piper-sample-generator>=3.2,<4"

!pip install -q \
    "mutagen>=1.47" \
    "torchinfo>=1.8" \
    "torchmetrics>=1.2" \
    "speechbrain>=0.5.14,<1" \
    "torch-audiomentations>=0.11,<1" \
    "acoustics>=0.2.6,<1" \
    "pronouncing>=0.2,<1" \
    "deep-phonemizer==0.0.19" \
    "datasets>=2.14,<4" \
    "webrtcvad>=2.0,<3" \
    "pyyaml>=6.0,<7"

import torch
print("torch:", torch.__version__,
      "cuda:", torch.cuda.is_available(),
      "device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")


## 1c — openWakeWord (patched) + LibriTTS generator + Piper shim (~1 min on H100)

Three things happen here:

1. **Clone openWakeWord at HEAD** (commit `368c037` or later — already
   moved tflite conversion behind a `--convert_to_tflite` flag and
   switched to `ai-edge-litert` for py3.12 support).
2. **Patch `setup.py`** to drop `speexdsp-ns` (no py3.12 wheel; the
   training pipeline does not use it — only deployment-time noise
   suppression that we don't run on Colab).
3. **Download the LibriTTS-R generator** (`.pt` from the v2.0.0 release
   + `.pt.json` config from the package's `models/` folder).
4. **Write a tiny shim file** at `/content/generate_samples.py` that
   re-exports the modern `piper_sample_generator.generate_samples` with
   the LibriTTS model path injected. openWakeWord's `train.py` does
   `from generate_samples import generate_samples` against
   `sys.path.insert(0, config["piper_sample_generator_path"])`, so by
   pointing `piper_sample_generator_path` at `/content` (where our shim
   lives) the trainer transparently picks up the new generator without
   any source-level patch to `train.py`.


In [ ]:
import os, sys, shutil
from pathlib import Path

# 1. Clone openWakeWord HEAD.
!rm -rf openwakeword
!git clone -q https://github.com/dscripka/openwakeword

# 2. Drop speexdsp-ns from install_requires — the only py3.12-incompatible
# entry in setup.py. Equivalent to ``training/wakeword/openwakeword_patches/setup.patch``.
!sed -i "/'speexdsp-ns/d" openwakeword/setup.py
!grep -n "install_requires\|speexdsp\|onnxruntime\|ai-edge" openwakeword/setup.py | head -8

# 3. Editable install — pulls onnxruntime + ai-edge-litert + scipy etc.
# We never call train.py with --convert_to_tflite, so the absence of
# tensorflow / tensorflow_probability / onnx_tf is fine.
!pip install -q -e ./openwakeword

# 4. LibriTTS generator (.pt) — 75 MB download.
LIBRITTS_DIR = Path("/content/piper_models")
LIBRITTS_DIR.mkdir(parents=True, exist_ok=True)
LIBRITTS_PT = LIBRITTS_DIR / "en_US-libritts_r-medium.pt"
LIBRITTS_JSON = Path(str(LIBRITTS_PT) + ".json")
if not LIBRITTS_PT.exists():
    !wget -q -O {LIBRITTS_PT} 'https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt'
# The companion .pt.json (phoneme map + speaker ids) ships in the repo, not the release.
if not LIBRITTS_JSON.exists():
    !wget -q -O {LIBRITTS_JSON} 'https://raw.githubusercontent.com/rhasspy/piper-sample-generator/master/models/en_US-libritts_r-medium.pt.json'
print("libritts pt :", LIBRITTS_PT.exists(), LIBRITTS_PT.stat().st_size, "bytes")
print("libritts cfg:", LIBRITTS_JSON.exists(), LIBRITTS_JSON.stat().st_size, "bytes")

# 5. Shim — translates openWakeWord's old `from generate_samples import generate_samples`
# into a call into the modern PyPI piper_sample_generator with the model path injected.
SHIM = Path("/content/generate_samples.py")
SHIM.write_text(
    "\"\"\"Drop-in replacement for the old piper-sample-generator script.\n\n"
    "openwakeword/train.py expects a `generate_samples(...)` function importable\n"
    "as `from generate_samples import generate_samples` after\n"
    "`sys.path.insert(0, piper_sample_generator_path)`. The modern PyPI package\n"
    "moved the function to `piper_sample_generator.generate_samples` and made\n"
    "the LibriTTS model path a required argument (`model=`); this shim bridges\n"
    "both deltas without touching openwakeword source.\n"
    "\"\"\"\n\n"
    "from piper_sample_generator import generate_samples as _real_generate_samples\n\n"
    f"_LIBRITTS_PT = \"{LIBRITTS_PT}\"\n\n"
    "def generate_samples(text, output_dir, max_samples=None, batch_size=1,\n"
    "                     file_names=None, slerp_weights=(0.5,),\n"
    "                     length_scales=(0.75, 1.0, 1.25),\n"
    "                     noise_scales=(0.667,), noise_scale_ws=(0.8,),\n"
    "                     max_speakers=None, **_ignored):\n"
    "    # `_ignored` swallows the openwakeword-only `auto_reduce_batch_size=True`\n"
    "    # kwarg, which the modern piper-sample-generator does not implement\n"
    "    # (pre-tune `tts_batch_size` in CONFIG to avoid OOM on smaller GPUs).\n"
    "    return _real_generate_samples(\n"
    "        text=text,\n"
    "        output_dir=output_dir,\n"
    "        model=_LIBRITTS_PT,\n"
    "        max_samples=max_samples,\n"
    "        batch_size=batch_size,\n"
    "        file_names=file_names,\n"
    "        slerp_weights=slerp_weights,\n"
    "        length_scales=length_scales,\n"
    "        noise_scales=noise_scales,\n"
    "        noise_scale_ws=noise_scale_ws,\n"
    "        max_speakers=max_speakers,\n"
    "    )\n"
)
print("wrote shim:", SHIM)


## 1d — openWakeWord feature models (mel-spec + speech embedding ONNX) (~5 s)

`train.py` calls `openwakeword.utils.AudioFeatures()` which defaults to
`inference_framework="onnx"` at HEAD — we only need the **ONNX** copies
of the melspectrogram and Google speech-embedding models, not the
`.tflite` versions.


In [ ]:
import os
os.makedirs("./openwakeword/openwakeword/resources/models", exist_ok=True)
for fname in ["embedding_model.onnx", "melspectrogram.onnx"]:
    target = f"./openwakeword/openwakeword/resources/models/{fname}"
    if not os.path.exists(target):
        !wget -q https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/{fname} -O {target}
    print(fname, ":", os.path.exists(target), os.path.getsize(target), "bytes")


## 1e — Imports for the rest of the notebook

In [ ]:
import os, sys, shutil, uuid, glob, json, time
from pathlib import Path
import numpy as np
import scipy, scipy.io.wavfile
import yaml
import datasets
from tqdm.auto import tqdm


## 2 — Mount Drive and stage user-recorded positives (~10 s)

Run `python -m record_samples` on your Mac first
(see `training/wakeword/README.md`), zip the resulting `positives/` folder,
and upload `positives.zip` to **`MyDrive/xiexie/`** before continuing.

If the zip is missing the notebook still trains a synthetic-only model —
quality drops noticeably (no real-mic, no real-room, no your-voice
positives) but the rest of the pipeline works.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

USER_POSITIVES_DIR = Path("/content/user_positives")
USER_POSITIVES_DIR.mkdir(exist_ok=True)

zip_path = Path(CONFIG["drive_positives_zip"])
if zip_path.exists():
    !unzip -q -o {zip_path} -d /content/user_positives_raw
    raw = Path("/content/user_positives_raw")
    wavs = list(raw.rglob("*.wav"))
    print(f"  {len(wavs)} user wav(s) found in {zip_path.name}")
    for w in wavs:
        shutil.copy(w, USER_POSITIVES_DIR / w.name)
else:
    print(f"  (no zip at {zip_path}) — synthetic-only positives will be used.")

print(f"staged {len(list(USER_POSITIVES_DIR.glob('*.wav')))} user wav(s)")


## 3a — MIT room-impulse-responses (~30 s)

These get convolved over the positives during augmentation, teaching the
model that "Xiexie" said in a noisy living room is still "Xiexie".


In [ ]:
rir_dir = Path("./mit_rirs")
rir_dir.mkdir(exist_ok=True)
if not list(rir_dir.glob("*.wav")):
    rirs = datasets.load_dataset(
        "davidscripka/MIT_environmental_impulse_responses",
        split="train", streaming=True,
    )
    for row in tqdm(rirs, desc="MIT RIRs"):
        name = row["audio"]["path"].split("/")[-1]
        scipy.io.wavfile.write(
            rir_dir / name, 16000,
            (row["audio"]["array"] * 32767).astype(np.int16),
        )
print("rirs:", len(list(rir_dir.glob("*.wav"))))


## 3b — AudioSet shard for background noise (~3 min on H100)

~1.5 GB tar, extracted to ./audioset/, then resampled to 16 kHz mono wavs.


In [ ]:
shard = CONFIG["audioset_shard"]
Path("audioset").mkdir(exist_ok=True)
tar_path = Path("audioset") / shard
if not tar_path.exists():
    !wget -q -O {tar_path} https://huggingface.co/datasets/agkphysics/AudioSet/resolve/main/data/{shard}
    !cd audioset && tar -xf {shard}

as16k = Path("./audioset_16k")
as16k.mkdir(exist_ok=True)
if len(list(as16k.glob("*.wav"))) < 100:
    flac_files = list(Path("audioset/audio").rglob("*.flac"))
    ds = datasets.Dataset.from_dict({"audio": [str(f) for f in flac_files]})
    ds = ds.cast_column("audio", datasets.Audio(sampling_rate=16000))
    for row in tqdm(ds, desc="AudioSet -> 16k wav"):
        name = Path(row["audio"]["path"]).stem + ".wav"
        scipy.io.wavfile.write(
            as16k / name, 16000,
            (row["audio"]["array"] * 32767).astype(np.int16),
        )
print("audioset_16k:", len(list(as16k.glob("*.wav"))))


## 3c — Free Music Archive — N hours of music (~2 min on H100)

Music is a useful additional negative-background signal because the
model also needs to *not* trigger on lyrics that contain "she/see"-like
phonemes.


In [ ]:
fma_dir = Path("./fma")
fma_dir.mkdir(exist_ok=True)
if len(list(fma_dir.glob("*.wav"))) < 50:
    fma = datasets.load_dataset(
        "rudraml/fma", name="small", split="train", streaming=True,
    )
    fma = iter(fma.cast_column("audio", datasets.Audio(sampling_rate=16000)))
    n_clips = CONFIG["fma_hours"] * 3600 // 30  # FMA-small clips are 30 s each.
    for _ in tqdm(range(n_clips), desc="FMA -> 16k wav"):
        try:
            row = next(fma)
        except StopIteration:
            break
        name = Path(row["audio"]["path"]).stem + ".wav"
        scipy.io.wavfile.write(
            fma_dir / name, 16000,
            (row["audio"]["array"] * 32767).astype(np.int16),
        )
print("fma:", len(list(fma_dir.glob("*.wav"))))


## 4 — Pre-computed openWakeWord features (~3 GB, ~2 min download on Colab)

~2 000 h of negative examples already passed through the Google
speech-embedding backbone — saves us re-extracting features at training
time. The validation set is ~11 h.


In [ ]:
if not Path("openwakeword_features_ACAV100M_2000_hrs_16bit.npy").exists():
    !wget -q https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/openwakeword_features_ACAV100M_2000_hrs_16bit.npy
if not Path("validation_set_features.npy").exists():
    !wget -q https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/validation_set_features.npy
print("feature files present:",
      Path("openwakeword_features_ACAV100M_2000_hrs_16bit.npy").exists(),
      Path("validation_set_features.npy").exists())


## 5 — Build the training YAML

openWakeWord's `train.py` is fully driven by a YAML file. We start from
the stock `examples/custom_model.yml` template and override the bits we
care about.

Critically, `piper_sample_generator_path` points at **`/content`** —
that's where our shim from § 1c lives, so the openwakeword trainer's
`from generate_samples import generate_samples` resolves to our wrapper.


In [ ]:
config = yaml.safe_load(open("openwakeword/examples/custom_model.yml").read())

config["model_name"]                       = CONFIG["model_name"]
config["target_phrase"]                    = [CONFIG["wake_word"]]
config["n_samples"]                        = CONFIG["n_samples_train"]
config["n_samples_val"]                    = CONFIG["n_samples_val"]
config["steps"]                            = CONFIG["steps"]
config["layer_size"]                       = CONFIG["layer_size"]
config["target_accuracy"]                  = CONFIG["target_accuracy"]
config["target_recall"]                    = CONFIG["target_recall"]
config["target_false_positives_per_hour"]  = CONFIG["target_false_positives_per_hour"]
config["max_negative_weight"]              = CONFIG["max_negative_weight"]
config["tts_batch_size"]                   = CONFIG["tts_batch_size"]

# /content holds our generate_samples.py shim — the openwakeword trainer
# does sys.path.insert(0, this_path); from generate_samples import ...
config["piper_sample_generator_path"]            = "/content"
config["rir_paths"]                              = ["./mit_rirs"]
config["background_paths"]                       = ["./audioset_16k", "./fma"]
config["background_paths_duplication_rate"]      = [1, 1]
config["false_positive_validation_data_path"]    = "validation_set_features.npy"
config["feature_data_files"]                     = {
    "ACAV100M_sample": "openwakeword_features_ACAV100M_2000_hrs_16bit.npy"
}
config["output_dir"]                             = "./my_custom_model"

Path("xiexie_model.yaml").write_text(yaml.safe_dump(config))
print("wrote xiexie_model.yaml")
print(yaml.safe_dump(config))


## 6 — Generate synthetic positives + adversarial negatives (~5 min on H100)

Piper synthesises N variants of "Xiexie" using the multi-speaker
LibriTTS model (varied speakers, length-scales, noise-scales) plus N
phoneme-overlapping adversarial phrases (computed with `pronouncing` +
`deep-phonemizer` — "Xiexie" is OOV for CMUDict, so DeepPhonemizer is
invoked once to convert it to phonemes, then the trainer generates
similar-sounding negatives).

H100 + libritts batch_size=64 ≈ 12–15 samples/s.


In [ ]:
!{sys.executable} openwakeword/openwakeword/train.py \
    --training_config xiexie_model.yaml --generate_clips


## 7 — Mix in the user's local recordings (~5 s)

We resample each user wav to 16 kHz mono int16 and copy
`user_recording_duplication` copies into `positive_train` (each augmented
separately downstream, producing many unique training examples from each
real recording). User recordings carry far more signal than synthetic
clips because they include the actual mic, room and voice the runtime
detector will see in the demo.


In [ ]:
import soundfile as sf
from scipy.signal import resample_poly

positive_train = Path("my_custom_model") / CONFIG["model_name"] / "positive_train"
positive_train.mkdir(parents=True, exist_ok=True)

user_wavs = sorted(USER_POSITIVES_DIR.glob("*.wav"))
duplicates = CONFIG["user_recording_duplication"]
TARGET_SR = 16000
TARGET_LEN_S = 1.5  # roughly the openWakeWord input window.
TARGET_LEN = int(TARGET_LEN_S * TARGET_SR)

added = 0
for w in user_wavs:
    audio, sr = sf.read(w, dtype="int16", always_2d=False)
    if audio.ndim > 1:
        audio = audio.mean(axis=1).astype(np.int16)
    if sr != TARGET_SR:
        audio = resample_poly(audio, TARGET_SR, sr).astype(np.int16)
    if len(audio) < TARGET_LEN:
        audio = np.pad(audio, (0, TARGET_LEN - len(audio)))
    elif len(audio) > TARGET_LEN:
        # Centre-crop so we don't lose the wake-word at either edge.
        start = (len(audio) - TARGET_LEN) // 2
        audio = audio[start:start + TARGET_LEN]
    for k in range(duplicates):
        out = positive_train / f"user_{w.stem}_{k:02d}_{uuid.uuid4().hex[:6]}.wav"
        sf.write(out, audio, TARGET_SR, subtype="PCM_16")
        added += 1

print(f"mixed in {added} user-recording copies "
      f"from {len(user_wavs)} unique wavs (x{duplicates}).")
print("positive_train total:", len(list(positive_train.glob('*.wav'))))


## 8a — Augment + extract features (~15 min on H100)

`--augment_clips` applies RIR convolutions and random noise from the
AudioSet/FMA shards, then runs the openWakeWord feature extractor on
every clip — features are memmapped during training so disk speed
matters.


In [ ]:
!{sys.executable} openwakeword/openwakeword/train.py \
    --training_config xiexie_model.yaml --augment_clips


## 8b — Train the small classifier (~30–60 min on H100)

`--train_model` runs the full auto-trainer (early stopping, checkpoint
averaging, cosine-decay LR, adaptive negative-weight schedule). Watch
the log: it prints validation accuracy / recall / false-pos-per-hour
every few hundred steps and stops when the targets in the YAML are met.


In [ ]:
!{sys.executable} openwakeword/openwakeword/train.py \
    --training_config xiexie_model.yaml --train_model


## 9 — Evaluate (recall on held-out positives + false-trigger on noise)

We score the freshly-exported ONNX model on:

* `positive_test/` — synthetic Piper positives held back from training (recall).
* the AudioSet 16 kHz wavs — definitely-not-"Xiexie" audio (false-trigger rate).

These are not perfect proxies for production performance — for that you'd
need real recordings of "Xiexie" and a long noise corpus from the
deployment environment — but they give a quick sanity signal.


In [ ]:
from openwakeword.model import Model as OwwModel

onnx_path = Path("my_custom_model") / CONFIG["model_name"] / f"{CONFIG['model_name']}.onnx"
assert onnx_path.exists(), f"expected trained model at {onnx_path}"

model = OwwModel(wakeword_models=[str(onnx_path)], inference_framework="onnx")
model_key = list(model.models.keys())[0]
THRESHOLD = 0.5

def score_clip(path):
    sr, audio = scipy.io.wavfile.read(path)
    if sr != 16000:
        from scipy.signal import resample_poly
        audio = resample_poly(audio, 16000, sr).astype(np.int16)
    if audio.ndim > 1:
        audio = audio.mean(axis=1).astype(np.int16)
    model.reset()
    chunk = 1280  # 80 ms
    peak = 0.0
    for i in range(0, len(audio) - chunk + 1, chunk):
        s = model.predict(audio[i:i + chunk])[model_key]
        if s > peak:
            peak = s
    return peak

# Recall on synthetic positive_test.
pos_test = Path("my_custom_model") / CONFIG["model_name"] / "positive_test"
pos_files = sorted(pos_test.glob("*.wav"))[:300]
pos_scores = [score_clip(p) for p in tqdm(pos_files, desc="positives")]
tp = sum(1 for s in pos_scores if s >= THRESHOLD)
fn = len(pos_scores) - tp
recall = tp / max(len(pos_scores), 1)

# False-trigger rate on AudioSet noise (definitely not "Xiexie").
neg_files = sorted(Path("audioset_16k").glob("*.wav"))[:200]
neg_scores = [score_clip(p) for p in tqdm(neg_files, desc="negatives")]
fp = sum(1 for s in neg_scores if s >= THRESHOLD)
tn = len(neg_scores) - fp
false_trigger_rate = fp / max(len(neg_scores), 1)

print()
print("=" * 60)
print(f"  threshold        : {THRESHOLD}")
print(f"  positives tested : {len(pos_scores)}")
print(f"    TP={tp}  FN={fn}  -> recall = {recall:.3f}")
print(f"  negatives tested : {len(neg_scores)}")
print(f"    FP={fp}  TN={tn}  -> false-trigger = {false_trigger_rate:.3f}")
print(f"  positive score: min={min(pos_scores):.3f}  median={np.median(pos_scores):.3f}  max={max(pos_scores):.3f}")
print(f"  negative score: min={min(neg_scores):.3f}  median={np.median(neg_scores):.3f}  max={max(neg_scores):.3f}")
print("=" * 60)


## 10 — Export `xiexie.onnx` to Drive

Drops the model in `MyDrive/xiexie/xiexie.onnx`. Download it locally and
place it at `<repo>/models/xiexie.onnx`, then restart the backend — the
runtime will pick it up automatically (see `backend/xiexie/voice/wake.py`).

Expected file size: **~1.3 MB** (a tiny FCN on top of the 384-dim
speech-embedding features).


In [ ]:
drive_dir = Path(CONFIG["drive_output_dir"])
drive_dir.mkdir(parents=True, exist_ok=True)

drive_onnx = drive_dir / f"{CONFIG['model_name']}.onnx"
shutil.copy(onnx_path, drive_onnx)

shutil.copy("xiexie_model.yaml", drive_dir / "xiexie_model.yaml")
(drive_dir / "eval_summary.json").write_text(json.dumps({
    "threshold": THRESHOLD,
    "recall": recall,
    "false_trigger_rate": false_trigger_rate,
    "positives_tested": len(pos_scores),
    "negatives_tested": len(neg_scores),
    "config": CONFIG,
}, indent=2))

print("exported:")
print(" -", drive_onnx, f"({drive_onnx.stat().st_size:,} bytes)")
print(" -", drive_dir / "xiexie_model.yaml")
print(" -", drive_dir / "eval_summary.json")
